# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

> https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Let's load the Croissant metadata and initialize our dataset using `mlcroissant`. We'll also preview the dataset's general information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object and pretty print the core information
meta = dataset.metadata
print(f"Title: {meta.name}")
print(f"Version: {meta.version if hasattr(meta, 'version') else 'N/A'}")
print(f"Description: {meta.description}")
print(f"License: {meta.license if hasattr(meta, 'license') else 'N/A'}")


## 2. Data Overview

We will review the available record sets, their `@id`s, associated fields, and columns as defined in the Croissant metadata. 
This helps us identify which parts of the data we can load for further analysis.


In [ ]:
# List all record sets defined in the dataset metadata
print("Record sets available:")
record_sets = []
if hasattr(meta, 'record_sets') or hasattr(meta, 'recordSet'):
    # Try both croissant 1.0 and croissant 1.x attribute names
    rs = getattr(meta, 'record_sets', None) or getattr(meta, 'recordSet', None)
    for record_set in rs:
        print(f"- @id: {record_set.id} | name: {record_set.name}")
        record_sets.append(record_set.id)
        # Show fields and columns
        if hasattr(record_set, 'fields') or hasattr(record_set, 'field'):
            fields = getattr(record_set, 'fields', None) or getattr(record_set, 'field', None)
            for field in fields:
                colinfo = f" (column: {getattr(field, 'column', getattr(field, 'columns', None))})" if hasattr(field, 'column') or hasattr(field, 'columns') else ""
                print(f"    - field @id: {field.id} | name: {field.name}{colinfo}")
else:
    print("No record sets found in the Croissant metadata.")

# In case nothing printed, note in markdown/cell below to show fallback instructions.

## 3. Data Extraction

Now, let's extract data from a record set and load it into a pandas DataFrame. We'll use the record set and field `@id`s we discovered above.


In [ ]:
# If record_sets is empty, manually set the main record set @id for this FAIR2 schema
if not record_sets:
    # As per Croissant schema best practices, if not listed, a common convention: use 'cr:RecordSet' or similar
    # However, the FAIR2 schema in this case is expected to have at least one main record set. Let's try common @ids.
    possible_ids = [
        'cr:RecordSet',  # Standard
        'https://api.app.sen.science/frontiers/7154140/recordset-1/recordset',
        'dv:ExperimentDataset',
    ]
    for pid in possible_ids:
        try:
            # Try to yield at least one record
            one_record = next(dataset.records(record_set=pid))
            record_sets = [pid]
            print(f"Auto-selected record set @id: {pid}")
            break
        except Exception:
            continue
if not record_sets:
    raise RuntimeError('Could not find any valid record set @id in metadata.')

# We'll load all available record sets
dataframes = {}
for rsid in record_sets:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for record set '@id': {rsid}")
        print(f"Fields (columns): {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {rsid}: {e}")

# For demonstration, let's assign the first available record set as the main one
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply some common EDA operations: filtering, normalization, grouping. 

For this dataset, let's pick a numeric field such as protein `coverage` or `MW` (molecular weight), using its `@id` or canonical column name, and demonstrate filtering and normalization. 


In [ ]:
# Inspect available columns
print("Columns in the main DataFrame:")
print(main_df.columns.tolist())

# Try to auto-detect a numeric field for demonstration
possible_numeric_fields = ['coverage', 'MW', 'peptide_count', 'abundance', 'intensity']
numeric_field = None
for field in possible_numeric_fields:
    if field in main_df.columns:
        numeric_field = field
        break
if numeric_field is None:
    # Fallback: pick the first float-like column
    for col in main_df.select_dtypes(include=['float', 'int']).columns:
        numeric_field = col
        break
print(f"Using numeric field: {numeric_field}")

# Define a threshold for filtering
threshold = 10  # demo threshold for filtering
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, normalized_col]].head())

# Try a grouping field (e.g. 'sample', 'organism', 'experiment'), pick first text field if unavailable
possible_group_fields = ['sample', 'experiment', 'condition', 'description', 'accession']
group_field = None
for gf in possible_group_fields:
    if gf in filtered_df.columns:
        group_field = gf
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization

Visualize the distribution of the numeric field and relationships in the filtered dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a group field exists, boxplot by group
if group_field:
    plt.figure(figsize=(10, min(7, 0.5*filtered_df[group_field].nunique()+2)))
    sns.boxplot(data=filtered_df, x=numeric_field, y=group_field, orient='h')
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to use `mlcroissant` to load, inspect, and analyze a research dataset defined by the [Croissant standard](https://mlcommons.org/croissant/). 

We:
- Loaded the FAIR² mass spectrometry dataset for human extracellular vesicles.
- Inspected record sets, fields, and explored available numeric variables.
- Performed simple EDA with filtering and normalization.
- Visualized data distributions to gain insights.

Further analysis might include advanced statistical modeling, exploring relationships among multiple fields, or integrating this dataset with others. For more details, see the Croissant schema or the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).